# Feature Engineering Report — Daily Revenue & COGS Forecasting

**Mục tiêu**: Xây dựng tập dữ liệu huấn luyện cho model dự báo Revenue & COGS theo ngày.

| Thông tin | Chi tiết |
|-----------|----------|
| **Primary Key** | `date` (mỗi ngày một dòng) |
| **Targets** | `revenue`, `cogs` |
| **Nguồn dữ liệu** | `daily_summary.parquet` + `features_daily_train.csv` + `reviews.csv` + `promotions.csv` |
| **Script** | `build_train.py` (v2 — leakage-safe) |

## 1. Tổng quan Pipeline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Load training dataset
df = pd.read_csv('dataset_cleaned/train_daily_forecasting.csv', parse_dates=['date'])
print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Features: {df.shape[1] - 3} (excl. date + 2 targets)")

## 2. Nguồn dữ liệu & Quy trình Merge

### Pipeline gồm 4 bước merge chính:

```
daily_summary.parquet (base: 46 cols)
    ├── LEFT JOIN features_daily_train.csv (+158 cols → calendar, fourier, lag, category shares)
    ├── LEFT JOIN reviews aggregated (+5 cols → daily_reviews, avg_rating, ...)
    └── LEFT JOIN promotions features (+6 cols → promo_active_flag, max_discount, ...)
        → Feature Engineering (leakage-safe)
        → Drop leaky columns (-83 cols)
        → Final: 193 cols (190 features + date + 2 targets)
```

In [ ]:
# Thống kê cơ bản targets
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(df['date'], df['revenue'], alpha=0.6, linewidth=0.8)
axes[0].set_title('Revenue theo ngày')
axes[0].set_ylabel('Revenue')

axes[1].plot(df['date'], df['cogs'], alpha=0.6, linewidth=0.8, color='orange')
axes[1].set_title('COGS theo ngày')
axes[1].set_ylabel('COGS')

plt.tight_layout()
plt.show()

print(f"\nRevenue — Mean: {df['revenue'].mean():,.0f} | Std: {df['revenue'].std():,.0f}")
print(f"COGS    — Mean: {df['cogs'].mean():,.0f} | Std: {df['cogs'].std():,.0f}")

## 3. Phân tích Data Leakage

### Nguyên tắc chống Leakage:
1. **Lag features**: Mọi feature tính từ dữ liệu thực tế PHẢI dùng `shift(1)` trở lên
2. **Rolling features**: Window PHẢI kết thúc tại ngày t-1 (dùng `.shift(1).rolling(w)`)
3. **Calendar & Promotion**: Deterministic (biết trước) → an toàn
4. **Inventory**: Dùng snapshot tháng trước → an toàn

### Các cột đã loại bỏ vì leakage:
- `gross_profit` (= revenue - cogs)
- `total_item_revenue` (≈ revenue)
- Tất cả cột cùng ngày: `total_orders`, `total_sessions`, `avg_rating`, ...
- Thay bằng version `_lag1` tương ứng

In [ ]:
# Kiểm tra correlation — không feature nào quá cao
feature_cols = [c for c in df.columns if c not in ['date', 'revenue', 'cogs']]
corr_rev = df[feature_cols].corrwith(df['revenue']).abs().sort_values(ascending=False)
corr_cogs = df[feature_cols].corrwith(df['cogs']).abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

corr_rev.head(20).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 20 Features — |Corr| với Revenue')
axes[0].invert_yaxis()

corr_cogs.head(20).plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Top 20 Features — |Corr| với COGS')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f"\nMax |corr| với Revenue: {corr_rev.max():.4f} ({corr_rev.idxmax()})")
print(f"Max |corr| với COGS:    {corr_cogs.max():.4f} ({corr_cogs.idxmax()})")
print("✅ Không có leakage (tất cả < 0.95)")

## 4. Nhóm Features Chi Tiết

In [ ]:
# Phân nhóm features
groups = {
    'Calendar/Temporal': [c for c in df.columns if any(k in c for k in
        ['year', 'month', 'quarter', 'day_of', 'week_of', 'is_weekend',
         'is_month', 'is_quarter', 'season', 'is_tet', 'is_1111', 'is_1212'])],
    'Fourier/Cyclical': [c for c in df.columns if any(k in c for k in
        ['sin_', 'cos_', 'yearly_sin', 'yearly_cos', 'weekly_sin', 'weekly_cos',
         'monthly_sin', 'monthly_cos'])],
    'Promotion (deterministic)': [c for c in df.columns if c.startswith('promo_active')
        or c.startswith('promo_max') or c.startswith('promo_avg')
        or c.startswith('promo_has_')],
    'Revenue Lag/Rolling': [c for c in df.columns if c.startswith('revenue_')],
    'COGS Lag/Rolling': [c for c in df.columns if c.startswith('cogs_')],
    'Margin Features': [c for c in df.columns if 'margin' in c or 'cogs_ratio' in c],
    'Order Features (lagged)': [c for c in df.columns if any(k in c for k in
        ['orders_lag', 'customers_lag', 'aov_', 'items_per_order', 'cancel_rate',
         'delivery_rate', 'n_orders_lag', 'n_orders_ma'])],
    'Discount (lagged)': [c for c in df.columns if any(k in c for k in
        ['discount_lag', 'promo_pct_lag', 'promo_pct_ma', 'promo_rate'])],
    'Web Traffic (lagged)': [c for c in df.columns if any(k in c for k in
        ['sessions_', 'visitors_', 'bounce_rate_lag', 'conversion_rate'])],
    'Returns (lagged)': [c for c in df.columns if any(k in c for k in
        ['return_rate', 'refund_lag'])],
    'Shipments (lagged)': [c for c in df.columns if any(k in c for k in
        ['shipments_lag', 'shipping_fee_lag'])],
    'Payments (lagged)': [c for c in df.columns if any(k in c for k in
        ['payment_value_lag', 'installments_lag'])],
    'Inventory': [c for c in df.columns if c.startswith('inv_')],
    'Reviews (lagged)': [c for c in df.columns if any(k in c for k in
        ['rating_', 'reviews_ma', 'avg_rating'])],
    'Category/Segment (lagged)': [c for c in df.columns if
        c.startswith('cat_share_') or c.startswith('seg_share_')],
}

summary_data = []
for name, cols in groups.items():
    if cols:
        summary_data.append({
            'Nhóm': name,
            'Số features': len(cols),
            'Leakage-safe': '✅',
            'Phương pháp': 'shift(1)' if 'lag' in name.lower() else 'deterministic' if 'Calendar' in name or 'Fourier' in name or 'Promotion' in name or 'Inventory' in name else 'shift(1)',
        })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print(f"\nTổng features: {sum(s['Số features'] for s in summary_data)}")

In [ ]:
# Hiển thị chi tiết từng nhóm
for name, cols in groups.items():
    if cols:
        print(f"\n{'='*60}")
        print(f"📦 {name} ({len(cols)} features)")
        print(f"{'='*60}")
        for c in sorted(cols):
            print(f"   • {c}")

## 5. Promotion Features — Deterministic (Biết trước)

Doanh nghiệp có **5 loại promotion cố định** mỗi năm với ngày tháng gần như không đổi:
- Spring Sale (~18/3 → 17/4)
- Mid-Year Sale (~23/6 → 22/7)
- Fall Launch (~30/8 → 1/10)
- Year-End Sale (~18/11 → 2/1)
- Urban Blowout / Rural Special (cách năm)

→ Features `promo_active_flag`, `promo_max_discount` KHÔNG bị leakage vì lịch biết trước.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(df['date'], df['promo_active_count'], linewidth=0.8)
ax.fill_between(df['date'], 0, df['promo_active_flag'], alpha=0.2, color='red', label='Promo Active')
ax.set_title('Promotion Active Periods')
ax.set_ylabel('Số promo active')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Revenue & COGS Lag Features — Leakage-Safe Pattern

**Pattern an toàn**: `df['feature'].shift(1).rolling(w).mean()`
- `shift(1)`: dịch dữ liệu 1 ngày → chỉ dùng quá khứ
- `rolling(w)`: window w ngày kết thúc tại t-1

**Lag depths**: 3, 7, 15, 30 (tránh chồng chéo)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Revenue lags
for lag in [3, 7, 15, 30]:
    col = f'revenue_lag_{lag}'
    if col in df.columns:
        axes[0,0].plot(df['date'].iloc[-180:], df[col].iloc[-180:], label=f'lag_{lag}', alpha=0.7)
axes[0,0].plot(df['date'].iloc[-180:], df['revenue'].iloc[-180:], 'k-', alpha=0.3, label='actual')
axes[0,0].set_title('Revenue Lag Features (last 180 days)')
axes[0,0].legend()

# Revenue MAs
for w in [3, 7, 15, 30]:
    col = f'revenue_ma_{w}'
    if col in df.columns:
        axes[0,1].plot(df['date'].iloc[-180:], df[col].iloc[-180:], label=f'MA_{w}', alpha=0.7)
axes[0,1].plot(df['date'].iloc[-180:], df['revenue'].iloc[-180:], 'k-', alpha=0.3, label='actual')
axes[0,1].set_title('Revenue Moving Averages (last 180 days)')
axes[0,1].legend()

# COGS lags
for lag in [3, 7, 15, 30]:
    col = f'cogs_lag_{lag}'
    if col in df.columns:
        axes[1,0].plot(df['date'].iloc[-180:], df[col].iloc[-180:], label=f'lag_{lag}', alpha=0.7)
axes[1,0].plot(df['date'].iloc[-180:], df['cogs'].iloc[-180:], 'k-', alpha=0.3, label='actual')
axes[1,0].set_title('COGS Lag Features (last 180 days)')
axes[1,0].legend()

# Margin features
if 'gross_margin_lag1' in df.columns:
    axes[1,1].plot(df['date'].iloc[-180:], df['gross_margin_lag1'].iloc[-180:], label='margin_lag1')
if 'gross_margin_ma7' in df.columns:
    axes[1,1].plot(df['date'].iloc[-180:], df['gross_margin_ma7'].iloc[-180:], label='margin_ma7')
if 'gross_margin_ma30' in df.columns:
    axes[1,1].plot(df['date'].iloc[-180:], df['gross_margin_ma30'].iloc[-180:], label='margin_ma30')
axes[1,1].set_title('Gross Margin Features (last 180 days)')
axes[1,1].legend()

plt.tight_layout()
plt.show()

## 7. Category/Segment Share Features

Mỗi segment có biên lợi nhuận khác nhau → thay đổi mix sẽ ảnh hưởng trực tiếp đến COGS.
Tất cả đều đã `shift(1)` để tránh leakage.

In [ ]:
cat_cols = [c for c in df.columns if c.startswith('cat_share_')]
seg_cols = [c for c in df.columns if c.startswith('seg_share_')]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

if cat_cols:
    df[cat_cols].iloc[-365:].plot(ax=axes[0], alpha=0.7, linewidth=0.8)
    axes[0].set_title('Category Shares (lagged, last 365 days)')
    axes[0].set_ylabel('Share')

if seg_cols:
    df[seg_cols].iloc[-365:].plot(ax=axes[1], alpha=0.7, linewidth=0.8)
    axes[1].set_title('Segment Shares (lagged, last 365 days)')
    axes[1].set_ylabel('Share')

plt.tight_layout()
plt.show()

## 8. Missing Values & Data Quality

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing': missing, '%': missing_pct})
missing_report = missing_report[missing_report['Missing'] > 0]

if len(missing_report) == 0:
    print("✅ Không có missing values trong dataset!")
else:
    print(f"⚠️ {len(missing_report)} cột có missing values:")
    print(missing_report.sort_values('%', ascending=False))

print(f"\nShape cuối cùng: {df.shape}")
print(f"Số features: {df.shape[1] - 3}")

## 9. Correlation Matrix — Top Features

In [ ]:
# Top correlated features with each target
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

top_rev = corr_rev.head(15)
top_cogs = corr_cogs.head(15)

sns.barplot(x=top_rev.values, y=top_rev.index, ax=axes[0], palette='Blues_d')
axes[0].set_title('Top 15 Features — |Corr| với Revenue')
axes[0].set_xlabel('|Correlation|')

sns.barplot(x=top_cogs.values, y=top_cogs.index, ax=axes[1], palette='Oranges_d')
axes[1].set_title('Top 15 Features — |Corr| với COGS')
axes[1].set_xlabel('|Correlation|')

plt.tight_layout()
plt.show()

## 10. Kết luận

### Training Dataset:
- **3,773 rows** × **193 columns** (190 features + date + 2 targets)
- **Date range**: 2012-09-02 → 2022-12-31
- **Leakage-safe**: Tất cả features từ dữ liệu thực tế đều dùng `shift(1)+`
- **Lag depths**: 3, 7, 15, 30 (tránh chồng chéo)

### Next Steps:
1. **Kiểm tra multicollinearity** giữa các features
2. **Feature selection** (loại bỏ features nhiễu/redundant)
3. **Train model** (XGBoost/LightGBM với TimeSeriesSplit)
4. **Evaluate** (MAE, RMSE, R² trên test set)
5. **SHAP analysis** để giải thích feature importance